In [ ]:
import pandas as pd
import re

# Load HZ sheet
hz = pd.read_excel(file_path, sheet_name="HZ")

# Machine columns
machine_cols = [col for col in hz.columns if col.startswith("Machine")]

# Get all unique machines
machines = pd.unique(hz[machine_cols].values.ravel())
machines = [m for m in machines if pd.notna(m)]

# Function to extract tonnage from machine name
def extract_machine_tonnage(machine):
    match = re.search(r'(\d+)T', machine)
    if match:
        return int(match.group(1))
    return None

machine_tonnage = {m: extract_machine_tonnage(m) for m in machines}

# Convert tonnage column into list
def parse_tonnage(x):
    if pd.isna(x):
        return []
    return [int(t.strip()) for t in str(x).split(",")]

hz["Tonnage_List"] = hz["Tonnage"].apply(parse_tonnage)

# Create compatibility matrix
matrix = []

for _, row in hz.iterrows():
    part = row["Part"]
    tonnage_list = row["Tonnage_List"]

    row_data = {"Part": part}

    for machine in machines:
        m_ton = machine_tonnage[machine]

        if m_ton in tonnage_list:
            row_data[machine] = 1
        else:
            row_data[machine] = 0

    matrix.append(row_data)

compatibility_matrix = pd.DataFrame(matrix)

print(compatibility_matrix)

In [ ]:
import pandas as pd
import re

# Load VT sheet
vt = pd.read_excel(file_path, sheet_name="VT")

# Machine columns
machine_cols = [col for col in vt.columns if col.startswith("Machine")]

# Extract all machines
machines = pd.unique(vt[machine_cols].values.ravel())
machines = [m for m in machines if pd.notna(m)]

# Function to extract tonnage from machine name
def extract_machine_tonnage(machine):
    match = re.search(r'(\d+)T', str(machine))
    if match:
        return int(match.group(1))
    return None

# Create machine → tonnage dictionary
machine_tonnage = {m: extract_machine_tonnage(m) for m in machines}

matrix = []

for _, row in vt.iterrows():

    part = row["Part"]
    part_tonnage = row["Tonnage"]

    row_data = {"Part": part}

    for machine in machines:

        if machine_tonnage[machine] == part_tonnage:
            row_data[machine] = 1
        else:
            row_data[machine] = 0

    matrix.append(row_data)

vt_matrix = pd.DataFrame(matrix)

print(vt_matrix)

In [ ]:
import pandas as pd
import re

# Load HZ sheet
file_path = "C:/Users/Ex0164/Book1.xlsx"
hz = pd.read_excel(file_path, sheet_name="HZ")

# Machine columns
machine_cols = [col for col in hz.columns if col.startswith("Machine")]

# Get all unique machines
machines = pd.unique(hz[machine_cols].values.ravel())
machines = [m for m in machines if pd.notna(m)]

# Function to extract tonnage from machine name
def extract_machine_tonnage(machine):
    match = re.search(r'(\d+)T', machine)
    if match:
        return int(match.group(1))
    return None

machine_tonnage = {m: extract_machine_tonnage(m) for m in machines}

# Convert tonnage column into list
def parse_tonnage(x):
    if pd.isna(x):
        return []
    tonnages = []
    for t in str(x).split(","):
        t = t.strip()
        match = re.search(r'(\d+)', t)
        if match:
            tonnages.append(int(match.group(1)))
    return tonnages

hz["Tonnage_List"] = hz["Tonnage"].apply(parse_tonnage)

# ==========================
# BUILD COMPATIBILITY MATRIX
# ==========================

part_machine_map = {}

for _, row in hz.iterrows():

    part = row["Part"]
    tonnage_list = row["Tonnage_List"]

    if part not in part_machine_map:
        part_machine_map[part] = {m:0 for m in machines}

    for machine in machines:

        m_ton = machine_tonnage[machine]

        if m_ton in tonnage_list:
            part_machine_map[part][machine] = 1

# Convert to dataframe
matrix_rows = []

for part, machine_dict in part_machine_map.items():
    row = {"Part":part}
    row.update(machine_dict)
    matrix_rows.append(row)

compatibility_matrix = pd.DataFrame(matrix_rows)

print(compatibility_matrix)

# Save compatibility matrix to Excel
output_path = "C:/Users/Ex0164/compatibility_matrix.xlsx"
compatibility_matrix.to_excel(output_path, index=False)

print(f"Compatibility matrix saved to {output_path}")

In [ ]:
import pandas as pd

# Files
vt_file = "C:/Users/Ex0164/Book1.xlsx"
unique_file = "C:/Users/Ex0164/unique_machines_vt.xlsx"

# Read VT sheet
vt = pd.read_excel(vt_file, sheet_name="VT")

# Read unique machines
unique = pd.read_excel(unique_file, sheet_name="Sheet1")

# Machine → tonnage from VT
machine_tonnage_vt = vt.set_index("Machine")["Tonnage"].to_dict()

# Machine → tonnage from unique machines file
machine_tonnage_unique = unique.set_index("Unique Machines")["Tonnage"].to_dict()

# Merge both mappings
machine_tonnage = {**machine_tonnage_vt, **machine_tonnage_unique}

# All machines
machines = list(machine_tonnage.keys())

matrix = []

for _, row in vt.iterrows():

    part = row["Part"]
    part_ton = row["Tonnage"]

    row_data = {"Part": part}

    for machine in machines:

        if machine_tonnage[machine] == part_ton:
            row_data[machine] = 1
        else:
            row_data[machine] = 0

    matrix.append(row_data)

vt_matrix = pd.DataFrame(matrix)

# Save
output_path = "C:/Users/Ex0164/compatibility_matrix.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    vt_matrix.to_excel(writer, sheet_name="VT_Matrix", index=False)

print("Compatibility matrix updated with new machines")

In [ ]:
import pandas as pd
import re

# =========================
# FILE PATHS
# =========================
vt_file = "C:/Users/Ex0164/Book1.xlsx"
unique_file = "D:/Tushar/TOOL FIX/Unique_Machines_vt.xlsx"

# =========================
# LOAD DATA
# =========================
vt = pd.read_excel(vt_file, sheet_name="VT")
unique = pd.read_excel(unique_file, sheet_name="Sheet1")

# =========================
# CLEAN COLUMNS
# =========================
vt.columns = vt.columns.str.strip()
unique.columns = unique.columns.str.strip()

# =========================
# CLEAN VT DATA
# =========================
vt = vt.dropna(subset=["Part", "Tons"]).copy()
vt["Part"] = vt["Part"].astype(str).str.strip()

# =========================
# EXTRACT PART TONNAGE LIST
# =========================
def extract_part_tons(text):
    # Handles: "40,120", "40 T, 120T", etc.
    numbers = re.findall(r'\d+', str(text))
    return list(set(int(n) for n in numbers))

vt["Tons_List"] = vt["Tons"].apply(extract_part_tons)

# =========================
# CLEAN MACHINE DATA
# =========================
unique = unique.dropna(subset=["Unique Machines", "Tonnage"]).copy()
unique["Unique Machines"] = unique["Unique Machines"].astype(str).str.strip()

# Extract machine tonnage (from "40T")
unique["Machine_Ton"] = unique["Tonnage"].apply(lambda x: int(re.search(r'\d+', str(x)).group()))

# =========================
# CREATE MACHINE → TON MAP
# =========================
machine_tonnage = dict(zip(unique["Unique Machines"], unique["Machine_Ton"]))

machines = list(machine_tonnage.keys())

# =========================
# BUILD COMPATIBILITY MATRIX
# =========================
matrix = []

for _, row in vt.iterrows():
    part = row["Part"]
    part_tons = row["Tons_List"]

    row_data = {"Part": part}

    for machine in machines:
        machine_ton = machine_tonnage[machine]

        # Compatibility check
        row_data[machine] = 1 if machine_ton in part_tons else 0

    matrix.append(row_data)

# =========================
# FINAL DATAFRAME
# =========================
compatibility_matrix = pd.DataFrame(matrix)

# =========================
# SAVE OUTPUT
# =========================
output_path = "C:/Users/Ex0164/compatibility_matrix.xlsx"
compatibility_matrix.to_excel(output_path, index=False)

print("✅ Compatibility matrix created successfully")

In [ ]:
import pandas as pd
import re

# =========================
# FILE PATHS
# =========================
file_path = "C:/Users/Ex0164/Book1.xlsx"
unique_file = "D:/Tushar/TOOL FIX/Unique_Machines_vt.xlsx"
output_path = "C:/Users/Ex0164/compatibility_matrix.xlsx"

# =========================
# HZ SHEET LOGIC
# =========================
hz = pd.read_excel(file_path, sheet_name="HZ")

# Machine columns in HZ
machine_cols = [col for col in hz.columns if col.startswith("Machine")]

# Get all unique machines
machines_hz = pd.unique(hz[machine_cols].values.ravel())
machines_hz = [m for m in machines_hz if pd.notna(m)]

# Extract tonnage from machine names
def extract_machine_tonnage(machine):
    match = re.search(r'(\d+)T', str(machine))
    if match:
        return int(match.group(1))
    return None

machine_tonnage_hz = {m: extract_machine_tonnage(m) for m in machines_hz}

# Parse comma-separated tonnage values
def parse_tonnage(x):
    if pd.isna(x):
        return []
    tonnages = []
    for t in str(x).split(","):
        t = t.strip()
        match = re.search(r'(\d+)', t)
        if match:
            tonnages.append(int(match.group(1)))
    return tonnages

hz["Tonnage_List"] = hz["Tonnage"].apply(parse_tonnage)

# Build HZ compatibility matrix
hz_matrix_data = []
for _, row in hz.iterrows():
    part = row["Part"]
    tonnage_list = row["Tonnage_List"]
    row_data = {"Part": part}
    for machine in machines_hz:
        m_ton = machine_tonnage_hz[machine]
        row_data[machine] = 1 if m_ton in tonnage_list else 0
    hz_matrix_data.append(row_data)

hz_matrix = pd.DataFrame(hz_matrix_data)

# =========================
# VT SHEET LOGIC
# =========================
vt = pd.read_excel(file_path, sheet_name="VT")
unique = pd.read_excel(unique_file, sheet_name="Sheet1")

# Clean columns
vt.columns = vt.columns.str.strip()
unique.columns = unique.columns.str.strip()

# Clean VT data
vt = vt.dropna(subset=["Part", "Tons"]).copy()
vt["Part"] = vt["Part"].astype(str).str.strip()

# Extract part tonnage list
def extract_part_tons(text):
    numbers = re.findall(r'\d+', str(text))
    return list(set(int(n) for n in numbers))

vt["Tons_List"] = vt["Tons"].apply(extract_part_tons)

# Clean machine data
unique = unique.dropna(subset=["Unique Machines", "Tonnage"]).copy()
unique["Unique Machines"] = unique["Unique Machines"].astype(str).str.strip()
unique["Machine_Ton"] = unique["Tonnage"].apply(lambda x: int(re.search(r'\d+', str(x)).group()))

# Machine → tonnage mapping
machine_tonnage_vt = dict(zip(unique["Unique Machines"], unique["Machine_Ton"]))
machines_vt = list(machine_tonnage_vt.keys())

# Build VT compatibility matrix
vt_matrix_data = []
for _, row in vt.iterrows():
    part = row["Part"]
    part_tons = row["Tons_List"]
    row_data = {"Part": part}
    for machine in machines_vt:
        machine_ton = machine_tonnage_vt[machine]
        row_data[machine] = 1 if machine_ton in part_tons else 0
    vt_matrix_data.append(row_data)

vt_matrix = pd.DataFrame(vt_matrix_data)

# =========================
# MACHINE PART COUNT (HZ)
# =========================
hz_numeric = hz_matrix.set_index("Part")
hz_machine_count = hz_numeric.sum(axis=0).reset_index()
hz_machine_count.columns = ["Machine", "Part_Count"]
hz_machine_count = hz_machine_count.sort_values(by="Part_Count", ascending=False)

# =========================
# MACHINE PART COUNT (VT)
# =========================
vt_numeric = vt_matrix.set_index("Part")
vt_machine_count = vt_numeric.sum(axis=0).reset_index()
vt_machine_count.columns = ["Machine", "Part_Count"]
vt_machine_count = vt_machine_count.sort_values(by="Part_Count", ascending=False)

# =========================
# SAVE ALL SHEETS
# =========================
with pd.ExcelWriter(output_path, engine="openpyxl", mode="w") as writer:
    hz_matrix.to_excel(writer, sheet_name="HZ_Matrix", index=False)
    vt_matrix.to_excel(writer, sheet_name="VT_Matrix", index=False)
    
    hz_machine_count.to_excel(writer, sheet_name="HZ_Machine_Part_Count", index=False)
    vt_machine_count.to_excel(writer, sheet_name="VT_Machine_Part_Count", index=False)

print("✅ All matrices and machine-part counts saved successfully!")